# Group Project - Safe routes home at night 

**DS105A Group Project – Data for Data Science (Autumn Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Group Notebook**
- 👤 Team Name: CtrlAltElite
- 🎯 Purpose: This notebook contains code outlining our data collection approach using the Google Maps Places API, Google Maps Routes API, and UK Police API . The full data collection pipeline is presented before the code.
</div>

In [1]:
import requests
import json
import os
from pathlib import Path
import warnings
import math
from collections import defaultdict
import re

## Data Collection Pipeline

1. Gather Nightclub Data
    - Requests Google Places API using textQuery for 3 areas in London. 
    - Validates the API response to ensure that returned locations are nightclubs/bars and that their postcodes are accurate.
    - Creates a subdir called `area_name` to store the nightclub data.

2. Gather Stations Data
    - Requests Google Places API per nightclub within each area. 
    - API fieldmask limits response to 20 stations within a 1 kilometre radius from a given nightclub. 
    - Creates a subdir called `stations` within each `area_name` directory to store the stations data.

3. Filter Stations Data 
    - Divides the area surrounding the club into eight 45 degree sections. 
    - For each section, takes the two closest stations to the nightclub **as well as** always including train, subway, and transit stations.
    - Creates a subdir within `stations` called `filtered_stations` to store the filtered stations data.

4. Gather Routes Data
    - Requests Google Routes API per nightclub-station pairing to generate routes data.
    - Creates a subdir within `area_name` called `routes` to store the routes data.

5. Prepare Routes Data for Police API
    - Reduces the number of route vertices present in the Google Routes API response polyline, preserving every third coordinate only. 
    - Converts the polyline into a polygon by extending the route polyline coordinates perpendicularly by 20m each side (total corridor width of 40m).
    - Converts the polyline coordinates into an acceptable format for Police API. 
    - Creates a separate metadata json preserving metadata for each route. File names for police_params and metadata correspond to one another e.g. `amazing_grace_london_bridge_route_0_metadata.json` and `amazing_grace_london_bridge_route_0_police_api_param.txt`.
    - Creates a `police_params` and `metadata` subdir within each `area_name` to store metadata and police params.

6. Gather crime data
    - Requests UK Police API for crime data within each route corridor for six months, from 2025-01 to 2025-06. 
    - Creates a `crime_data` subdir within each `area_name` which has a subdir for each month's crime data e.g. `2025-01`

## 1. Gather nightclub data for areas in London

In [3]:
# Constants
PLACES_URL = 'https://places.googleapis.com/v1/places:searchText'
API_KEY = os.getenv('API_KEY')
HEADERS = {
    'Content-Type': 'application/json',
    'X-Goog-API-Key': API_KEY,
    'X-Goog-FieldMask': (
        'places.displayName,'
        'places.postalAddress.postalCode,'
        'places.primaryType,'
        'places.location'
    )
}

AREA_NAMES = ['Dalston', 'Hackney_Wick', 'Bermondsey']
AREA_POSTCODES = {
    'Dalston': ('E8', 'N16'),
    #https://www.townscountiespostcodes.co.uk/towns-in-england/london/dalston/
    'Hackney_Wick': ('E9', 'E15', 'E20'),
    #https://www.townscountiespostcodes.co.uk/postcodes-in-england/london/hackney-wick/
    # https://www.doogal.co.uk/AdministrativeAreas?ward=E05009374
    'Bermondsey': ('SE1', 'SE16')
    # https://www.townscountiespostcodes.co.uk/postcodes-in-england/london/bermondsey/
}


In [ ]:
# define a function to normalize area names
def area_slugify(name: str) -> str:
    return re.sub(r'[\s\-]+', '_', name.strip().lower())

In [ ]:
def validate_clubs(area_name):
    """
    Queries Google Places API for nightclubs within a given area, 
    validates them by checking that their primaryType is either a bar 
    or a nightclub and that they postcode is correct, caches results, 
    and returns the filtered list. It creates a subdir to store data by area.
    """

    area_slug = area_slugify(area_name)
    query_area = area_name.replace('_', '')

    cache_dir = Path(f'data/raw/{area_slug}')
    cache_path = cache_dir / (f'{area_slug}_nightclubs.json')

    try: 

        query = {
            'textQuery': f'nightclubs in {query_area}, London, England'
        }

        response = requests.post(url=PLACES_URL, headers=HEADERS, data=json.dumps(query))

        response.raise_for_status()
        clubs_data = response.json()
    
    except Exception as e:
        if cache_path.exists():
            warnings.warn(
                f'Places API failed for {area_name}, using cached data instead'
            )
            with open(cache_path) as f:
                return json.load(f)

        else:
            raise RuntimeError(
                f'API failed and no cache available for {area_name}'
            ) from e

    # Validate clubs returned from API against their primaryType and postcode

    valid_clubs = []
    postcode_prefixes = AREA_POSTCODES.get(area_name)
    if not postcode_prefixes:
        raise ValueError(f'No postcode prefixes defined for {area_name}')

    for club in clubs_data.get('places', []):
        postcode = (
            club
            .get('postalAddress', {})
            .get('postalCode', '')
        )
        primary_type = club.get("primaryType", "")

        if (
            postcode.startswith(postcode_prefixes)
            and primary_type in {"bar", "nightclub"}
        ):
            valid_clubs.append(club)

    cache_dir.mkdir(parents=True, exist_ok=True)
    with open(cache_path, 'w') as f:
            json.dump(valid_clubs, f, indent=2)

    return valid_clubs

In [16]:
for area_name in AREA_NAMES:
    validate_clubs(area_name)

## 2. Collect up to twenty stations within a radius of 1 kilometre from the nightclub

In [3]:
# Test constants

API_KEY = os.getenv('API_KEY')
STATIONS_URL = 'https://places.googleapis.com/v1/places:searchNearby'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'places.types,places.formattedAddress,places.location,places.displayName,places.primaryTypeDisplayName',
    }
ROUNDING = 4 # round location data to avoid unneccesary API calls for similar locations - this allows for roughly a 10m distance 

In [5]:
# define a function to normalize club names for a coherent file system 
def slugify(name: str) -> str:
    name = name.lower()
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name.strip("_")

In [ ]:
def get_stations_data(area_name):

    """
    Requests google places API for stations within 1000m of a nightclub.
    Stores all stations for one club in one json file
    """

    area_slug = area_slugify(area_name)

    clubs_path = Path(f'data/raw/{area_slug}/{area_slug}_nightclubs.json')

    if not clubs_path.exists():
        raise FileNotFoundError(
            f'No nightclub cache found for {area_name} at {clubs_path}'
        )
    
    with open(clubs_path, 'r') as f:
        clubs = json.load(f)

    output_dir = Path(f"data/raw/{area_slug}/stations")
    output_dir.mkdir(parents=True, exist_ok=True)

    for club in clubs:
        club_name = club['displayName']['text']
        club_slug = slugify(club_name)

        lat = round(club['location']['latitude'], ROUNDING)
        lon = round(club['location']['longitude'], ROUNDING)

        cache_path = output_dir / f'{club_slug}_stations.json'

        if cache_path.exists():
            continue

        data = {
                'includedTypes' : ['bus_station', 'subway_station', 'train_station'],
                'maxResultCount' : 20,
                'rankPreference': 'DISTANCE',
                'locationRestriction' : {
                    'circle': {
                        'center' : {
                            'latitude' : lat,
                            'longitude' : lon
                        },
                        'radius' : 1000
                    }
                }
            }
            

        try: 
            response = requests.post(url=STATIONS_URL, headers=HEADERS, json=data, timeout=10)
            response.raise_for_status()
            stations_data = response.json()

        except Exception as e:
            warnings.warn(
                f'Failed to fetch stations for {club_name}: {e}'
            )
            continue

        output = {
            'area': area_name,
            'club': club_name,
            'club_location': {
                'latitude': lat,
                'longitude': lon,
            },
            'places': stations_data.get('places', []),
        }

        with open(cache_path, 'w') as f:
            json.dump(output, f, indent=2)

In [18]:
for area_name in AREA_NAMES:
    get_stations_data(area_name)

## 3. Filter stations according to direction

In [ ]:
# define a function that returns a bearing from point 1 to point 2 in degrees (0-360) where 0 = North and 90 = East

def bearing_degrees(lat1, lon1, lat2, lon2):
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_lon = math.radians(lon2 - lon1)

    x = math.sin(delta_lon) * math.cos(phi2)
    y = (
        math.cos(phi1) * math.sin(phi2)
        - math.sin(phi1) * math.cos(phi2) * math.cos(delta_lon)
    )

    bearing = math.degrees(math.atan2(x, y))
    return (bearing + 360) % 360

In [20]:
# define a function that assigns each station to one of 8 sectors

def sector_index(bearing):
    return int(bearing // 45)

In [6]:
# define haversine to calculate the shortest distance between two points on a sphere

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

In [ ]:
def directional_filtering(stations, club_lat, club_lon, max_per_sector=2):

    """
    Divides the area around a club into 8 sectors, taking a maximum of 2 
    stations per sector. Automatically accepts train stations, subway 
    stations, and transit stations.
    """
    
    sectors = defaultdict(list)
    train_stations = []

    for station in stations['places']:
        lat = station['location']['latitude']
        lon = station['location']['longitude']

        # 1. compute straight-line distance in metres from club to station

        station['distance_m'] = haversine(club_lat, club_lon, lat, lon)

        # 2. assign station to a bearing sector relative to the club

        bearing = bearing_degrees(club_lat, club_lon, lat, lon)
        sector = sector_index(bearing)

        station['bearing'] = bearing
        station['sector'] = sector
        
        # 3. split by station type

        if station['primaryType'][0] in ('train_station', 'subway_station', 'transit_station'):
            train_stations.append(station)
        else:
            sectors[sector].append(station)

    selected = []

    # select closest bus stations per sector

    for sector_stations in sectors.values():
        sector_stations.sort(key=lambda s: s['distance_m'])
        selected.extend(sector_stations[:max_per_sector])

    # always include train/subway stations

    selected_ids = {
        (
            s['location']['latitude'], s['location']['longitude']
            ) 
            for s in selected
        }

    for station in train_stations:
        key = (station['location']['latitude'], station['location']['longitude'])
        if key not in selected_ids:
            selected.append(station)

    return selected

In [ ]:
def filter_stations_for_area(area_name, max_per_sector=2):
    """
    Filters stations per club per area.
    """
    
    area_slug = area_slugify(area_name)
    stations_dir = Path(f'data/raw/{area_slug}/stations')

    if not stations_dir.exists():
        raise FileNotFoundError(f'No station directory found at {stations_dir}')

    output_dir = stations_dir / 'filtered_stations'
    output_dir.mkdir(parents=True, exist_ok=True)

    for station_file in stations_dir.glob('*_stations.json'):
        with open(station_file) as f:
            data = json.load(f)

        club_lat = data['club_location']['latitude']
        club_lon = data['club_location']['longitude']

        try:
            filtered = directional_filtering(
                stations=data,
                club_lat=club_lat,
                club_lon=club_lon,
                max_per_sector=max_per_sector,
            )
        except Exception as e:
            warnings.warn(
                f'Filtering failed for {station_file.name}: {e}'
            )

            continue 

        output = {
            'area': data['area'],
            'club': data['club'],
            'club_location': data['club_location'],
            'selected_stations': filtered,
        }

        out_path = output_dir / station_file.name.replace(
            '_stations.json', '_stations_filtered.json'
        )

        with open(out_path, 'w') as f:
            json.dump(output, f, indent=2)


In [37]:
for area_name in AREA_NAMES:
    filter_stations_for_area(
        area_name = area_name,
        max_per_sector=2
    )

## 4. Generate Routes for Stations

In [11]:
# Test constants

API_KEY = os.getenv('API_KEY')
ROUTES_URL = 'https://routes.googleapis.com/directions/v2:computeRoutes'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'routes.duration,routes.distanceMeters,routes.polyline',
    }

In [ ]:
def collect_routes_for_club(area_name, club_slug):

    """
    Calls google routes api for each club-station route.
    """

    area_slug = area_slugify(area_name)
    base_dir = Path(f'data/raw/{area_slug}')

    filtered_stations_dir = base_dir / 'stations' / 'filtered_stations'
    stations_path = filtered_stations_dir/ f'{club_slug}_stations_filtered.json'
    
    routes_dir = base_dir / 'routes'
    routes_dir.mkdir(parents=True, exist_ok=True)
    routes_path = routes_dir / f'{club_slug}_routes.json'

    if not stations_path.exists():
        raise FileNotFoundError(f'Missing stations file {stations_path}')

    with open(stations_path, 'r') as f:
        stations_data = json.load(f)

    club_lat = stations_data['club_location']['latitude']
    club_lon = stations_data['club_location']['longitude']

    routes_output = {
    'area': area_name,
    'club': stations_data['club'],
    'club_location': stations_data['club_location'],
    'routes': []
    }

    for station in stations_data.get('selected_stations', []):

        station_lat = station['location']['latitude']
        station_lon = station['location']['longitude']

        data = {
            'origin': {
                'location': {
                    'latLng': {
                        'latitude': club_lat,
                        'longitude': club_lon
                    }
                }
            },
            'destination': {
                'location': {
                    'latLng': {
                        'latitude': station_lat,
                        'longitude': station_lon
                    }
                }
            },
            'travelMode': 'WALK',
            'polylineEncoding': 'GEO_JSON_LINESTRING',
            'polylineQuality': 'OVERVIEW'
        }

        try:
            response = requests.post(
                ROUTES_URL,
                headers=HEADERS,
                json=data,
                timeout=10
            )
            response.raise_for_status()
            # Save one route per journey
            route = response.json()['routes'][0]

            routes_output['routes'].append({
                'station_name': station['displayName']['text'],
                'station_location': station['location'],
                'distanceMeters': route['distanceMeters'],
                'duration': route['duration'],
                'polyline': route['polyline']['geoJsonLinestring']['coordinates']
            })

        except Exception as e:
            warnings.warn(
                f'Route failed for {stations_data['club']}'
                f'{station['displayName']['text']}: {e}'
            )

    # Cache routes separately
    with open(routes_path, 'w') as f:
        json.dump(routes_output, f, indent=2)

    return routes_output

In [12]:
area_club_slugs = {}

for area_name in AREA_NAMES:
    area_slug = area_slugify(area_name)
    filtered_dir = Path(f'data/raw/{area_slug}/stations/filtered_stations')
    
    if not filtered_dir.exists():
        warnings.war(f'No filtered stations directory for {area_name}')
        continue
    
    area_club_slugs[area_name] = []

    for file_path in filtered_dir.iterdir():
        if file_path.is_file():
            club_slug = file_path.name.replace('_stations_filtered.json', '')
            area_club_slugs[area_name].append(club_slug)

for area_name, club_slugs in area_club_slugs.items():
    for club_slug in club_slugs:
        collect_routes_for_club(area_name, club_slug)

## 5. Prepare Routes for Police API

In [13]:
def build_police_api_param(
    route,
    club_name,
    area_name,
    step=3,
    corridor_width_m=20,
):

    """
    Builds a Police API polygon param from a Google Routes API response.
    """
    # First, reduce the number of polyline coordinates for good API citizenship when querying the Police API.

    polyline = route['polyline']
    sampled = polyline[::step]

    if sampled[-1] != polyline[-1]:
        sampled.append(polyline[-1])

    # Next, build a corridor polygon from the polyline coords

    left_side = []
    right_side = []

    for i in range(len(sampled) - 1):
        # coordinates switched because Google Routes returns coords as [longitude, latitude]
        lon1, lat1 = sampled[i]
        lon2, lat2 = sampled[i + 1]

        metres_per_deg_lat = 111_320
        metres_per_deg_lon = 111_320 * math.cos(math.radians(lat1))

        dx = lon2 - lon1
        dy = lat2 - lat1

        length = math.hypot(
            dx * metres_per_deg_lon,
            dy * metres_per_deg_lat,
        )

        # extend coordinates laterally to extend the route into a corridor

        ux = -dy * metres_per_deg_lat / length
        uy =  dx * metres_per_deg_lon / length

        offset_lon = (ux * corridor_width_m) / metres_per_deg_lon
        offset_lat = (uy * corridor_width_m) / metres_per_deg_lat

        left_side.append([lat1 + offset_lat, lon1 + offset_lon])
        right_side.append([lat1 - offset_lat, lon1 - offset_lon])

    # close polygon

    lon_end, lat_end = sampled[-1]
    left_side.append([lat_end + offset_lat, lon_end + offset_lon])
    right_side.append([lat_end - offset_lat, lon_end - offset_lon])

    polygon = left_side + right_side[::-1]
    polygon.append(polygon[0])

    # Finally, edit polygon to match Police API format

    param = ":".join(f"{lat},{lon}" for lat, lon in polygon)

    return param

In [ ]:
def build_route_metadata(
    *,
    area_name: str,
    club_name: str,
    club_location: dict,
    route: dict,
    route_index: int, 
    police_param_filename: str,
):
    """
    Create a metadata json for each police API param txt file.
    """
    # convert data such as 55s into int seconds 

    duration_raw = route.get("duration", "")
    duration_seconds = (
        int(duration_raw.rstrip("s"))
        if isinstance(duration_raw, str) and duration_raw.endswith("s")
        else None
    )

    metadata = {
        "area": area_name,
        "club": {
            "name": club_name,
            "location": {
                "latitude": club_location["latitude"],
                "longitude": club_location["longitude"],
            },
        },
        "destination": {
            "name": route["station_name"],
            "location": {
                "latitude": route["station_location"]["latitude"],
                "longitude": route["station_location"]["longitude"],
            },
        },
        "route": {
            "route_index": route_index,
            "distance_meters": route.get("distanceMeters"),
            "duration_seconds": duration_seconds,
        },
        "police_api": {
            "param_file": police_param_filename,
        },
    }

    return metadata

In [16]:
def batch_build_police_api_params(
    area_name,
    corridor_width_m,
):
    """
    Batch-builds Police API polygon params for all clubs in an area directory.
    Saves param as a text file, one file per route
    """

    area_dir = Path(f'data/raw/{area_name.lower()}')
    routes_dir = area_dir / 'routes'
    if not routes_dir.exists():
        raise FileNotFoundError(f'Routes directory not found: {routes_dir}')
    
    routes_files = list(routes_dir.glob("*_routes.json"))

    if not routes_files:
        raise RuntimeError(f"No route files found in {routes_dir}")

    police_params_dir = area_dir / 'police_params'
    metadata_dir = area_dir / 'metadata'

    police_params_dir.mkdir(exist_ok=True)
    metadata_dir.mkdir(exist_ok=True)

    results = {}

    for routes_path in routes_files:
        club_slug = routes_path.stem.replace("_routes", "")
        club_name = club_slug.replace("_", " ").title()

        with open(routes_path) as f:
            data = json.load(f)

        if 'routes' not in data or not data['routes']:
            warnings.warn(f'No routes found in {routes_path.name}')
            continue

        club_location = data.get('club_location')

        for i, single_route in enumerate(data['routes']):

            param_filename = f'{club_slug}_route_{i}_police_api_param.txt'
            param_path = police_params_dir/param_filename

            metadata_path = metadata_dir/f'{club_slug}_route_{i}_metadata.json'

            try:
                param = build_police_api_param(
                route=single_route,
                club_name=club_name,
                area_name=area_name,
                corridor_width_m=corridor_width_m,
                )

                param_path.write_text(param)

                metadata = build_route_metadata(
                    area_name=area_name,
                    club_name=club_name,
                    club_location=club_location,
                    route=single_route,
                    route_index=i,
                    police_param_filename=param_filename,
                )

                with open(metadata_path, 'w') as f:
                    json.dump(metadata, f, indent=2)

                results[f"{club_name} | route {i}"] = {
                    "param_file": str(param_path),
                    "metadata_file": str(metadata_path),
                }

            except Exception as e:
                warnings.warn(
                    f"Failed to build Police API param for {club_name}: {e}"
                )

    return results

In [17]:
for area_name in AREA_NAMES:
    batch_build_police_api_params(
        area_name=area_name,
        corridor_width_m=40,
    )

## 6. Request Police API

In [19]:
def call_police_api(area_name, date):

    """
    Calls the UK Police API for each saved route polygon in an area,
    saving filtered crime data per route.
    """

    area_dir = Path(f'data/raw/{area_name.lower()}')
    params_dir = area_dir / 'police_params'

    if not params_dir.exists():
        raise FileNotFoundError(f"Area directory not found: {area_dir}")

    param_files = list(params_dir.glob("*_police_api_param.txt"))

    if not param_files:
        raise RuntimeError(f"No police API files found in {params_dir}")

    POLICE_URL = "https://data.police.uk/api/crimes-street/all-crime"

    allowed_categories = {
            "anti-social-behaviour",
            "violent-crime",
            "public-order",
            'sexual-offences',
        }

    output_dir = area_dir / 'crime_data' / date
    output_dir.mkdir(parents=True, exist_ok=True)

    for param_path in param_files:

        route_id = param_path.stem.replace('_police_api_param', '')   
        
        with open(param_path, 'r') as f:
            poly = f.read()

        params = {
            'date': date,
            'poly': poly,
        }
        
        response = requests.get(POLICE_URL, params=params)

        if response.status_code != 200:
            warnings.warn(
                f"Police API failed for {route_id}: {response.status_code}"
            )
            continue

        crimes = response.json()

        filtered_crimes = [
            c for c in crimes
            if c.get("category") in allowed_categories
        ]

        output_path = output_dir / f"{route_id}_crime_data_{date}.json"

        with open(output_path, 'w') as f:
            json.dump(filtered_crimes, f, indent=2)

In [21]:
dates = ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06']

for area in AREA_NAMES:
    for date in dates:
        try: 
            print(f'Fetching crimes for {area} - {date}')
            call_police_api(area_name=area, date=date)
        except Exception as e:
            warnings.warn(
                f'Crime collection failed for {area} ({date}): {e}'
            )

Fetching crimes for Dalston - 2025-01
Fetching crimes for Dalston - 2025-02
Fetching crimes for Dalston - 2025-03
Fetching crimes for Dalston - 2025-04
Fetching crimes for Dalston - 2025-05
Fetching crimes for Dalston - 2025-06
Fetching crimes for Hackney_Wick - 2025-01
Fetching crimes for Hackney_Wick - 2025-02
Fetching crimes for Hackney_Wick - 2025-03
Fetching crimes for Hackney_Wick - 2025-04
Fetching crimes for Hackney_Wick - 2025-05
Fetching crimes for Hackney_Wick - 2025-06
Fetching crimes for Bermondsey - 2025-01
Fetching crimes for Bermondsey - 2025-02
Fetching crimes for Bermondsey - 2025-03
Fetching crimes for Bermondsey - 2025-04
Fetching crimes for Bermondsey - 2025-05
Fetching crimes for Bermondsey - 2025-06


## 7. Check Night Service for Stations

(check which routes have Night Tube or Night bus service using TfL API. Routes without night service will be filtered out in NB02 (Data Transformation))

In [ ]:
# TfL API Configuration
TFL_BASE_URL = "https://api.tfl.gov.uk"

def area_slugify(name: str) -> str:
    """Normalize area names for coherent file system (matches existing function)."""
    return name.strip().lower().replace(' ', '_').replace('-', '_')

def query_tfl_api(endpoint, params=None):
   
    url = f"{TFL_BASE_URL}{endpoint}"
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        warnings.warn(f"TfL API error for {endpoint}: {e}")
        return None
    except Exception as e:
        warnings.warn(f"Unexpected error: {e}")
        return None

💭 **Personal Reflection:**
- a "slug" is a URL friendly string or text and to "slugify" is to convert a text or string like a title or phrase into a "slug" by making it lowercase, removing special characters and replacing spaces with hyphens. 
- While Eli has used caching in his API calls above, I have not used caching here because TfL API does not require an API key (unlike Google Places API), has simple endpoints and data changes more, with night service routes being added/removed, compared to nightclub locations which are stable.

In [ ]:
def find_nearest_stop_points(station_lat, station_lon, radius=150):
    """
    Find TfL stop point IDs near station coordinates. 
    TfL needs stop point IDs to query night service (e.g. 940GZZLULNB (London Bridge Underground). 
    A station may be recorded as 51.5050 in our station_data but recorded as 51.5052, hence collecting stop points within a small radius of 150m to allow for variation in coordinates between sources.
    A single station may have multiple stop points (tube platforms, bus stops, etc.)
    """
    endpoint = f"/StopPoint"
    params = {
        'lat': station_lat,
        'lon': station_lon,
        'stopTypes': 'NaptanMetroStation,NaptanRailStation,NaptanBusCoachStation,NaptanPublicBusCoachTram',
        'radius': radius
    }
    
    data = query_tfl_api(endpoint, params)
    
    if not data or 'stopPoints' not in data:
        return []
    
    stop_points = []
    for stop in data['stopPoints']:
        stop_points.append({
            'id': stop.get('naptanId', ''),
            'name': stop.get('commonName', ''),
            'modes': stop.get('modes', []),
            'lines': [line.get('name', '') for line in stop.get('lines', [])]
        })
    
    return stop_points

In [ ]:
def check_stop_night_service(stop_id):
    """
    Check if a TfL stop point has night service.
    Uses both ServiceTypes endpoint and Route filtering.
    
    Args:
        stop_id: TfL stop point ID (naptanId)
    
    Returns:
        Dictionary with night service information
    """
    # Query the service types for this stop
    endpoint = f"/StopPoint/{stop_id}/ServiceTypes"
    
    data = query_tfl_api(endpoint)
    
    has_night_service_type = False
    service_types = []
    
    if data:
        if isinstance(data, list):
            service_types = data
            has_night_service_type = 'Night' in data or 'night' in [s.lower() for s in data]
        elif isinstance(data, dict) and 'serviceTypes' in data:
            service_types = data['serviceTypes']
            has_night_service_type = 'Night' in service_types or 'night' in [s.lower() for s in service_types]
    
    # Also check for routes with night service filter
    # This catches 24-hour services that may not have "Night" type but do run at night
    route_endpoint = f"/StopPoint/{stop_id}/Route"
    route_params = {'serviceTypes': 'Night'}
    route_data = query_tfl_api(route_endpoint, route_params)
    
    has_night_routes = route_data and len(route_data) > 0
    
    # COMBINED check: has explicit night service type OR has routes that run at night
    has_night = has_night_service_type or has_night_routes
    
    return {
        'has_night_service': has_night,
        'has_night_service_type': has_night_service_type,
        'has_night_routes': has_night_routes,
        'service_types': service_types,
        'method': 'combined_check'
    }

def get_night_routes_at_stop(stop_id):
    """
    Get specific night routes at a stop (tube lines and bus routes).
    Uses TfL API exclusively - no hardcoded lists.
    
    Args:
        stop_id: TfL stop point ID
    
    Returns:
        Dictionary with night tube lines, night bus routes, and 24hr buses
    """
    # Query routes with night service type
    endpoint = f"/StopPoint/{stop_id}/Route"
    params = {'serviceTypes': 'Night'}
    
    night_data = query_tfl_api(endpoint, params)
    
    night_tube_lines = set()
    night_bus_routes = set()  # N-prefixed routes
    regular_buses_with_night_service = set()  # Regular numbered routes that run at night
    
    if night_data:
        for route in night_data:
            line_name = route.get('name', '')
            mode = route.get('modeName', '').lower()
            
            if mode == 'tube':
                night_tube_lines.add(line_name)
            elif mode == 'bus':
                if line_name.startswith('N'):
                    night_bus_routes.add(line_name)
                else:
                    # Regular numbered route that appears in night service query (24hr buses)
                    regular_buses_with_night_service.add(line_name)
    
    return {
        'night_tube_lines': sorted(list(night_tube_lines)),
        'night_bus_routes': sorted(list(night_bus_routes)),
        'regular_buses_with_night_service': sorted(list(regular_buses_with_night_service))
    }

def analyze_station_night_service(station_data):
    """
    Analyze night service availability for a station.
    
    Args:
        station_data: Station dict with location, displayName, primaryType
    
    Returns:
        Dictionary with complete night service analysis
    """
    station_name = station_data.get('displayName', [''])[0].get('text', '') if isinstance(station_data.get('displayName'), list) else station_data.get('displayName', '')
    station_lat = station_data['location']['latitude']
    station_lon = station_data['location']['longitude']
    station_type = station_data.get('primaryType', [{}])[0] if isinstance(station_data.get('primaryType'), list) else station_data.get('primaryType', '')
    
    print(f"        Analyzing: {station_name}")
    
    # Find nearest stop points
    stop_points = find_nearest_stop_points(station_lat, station_lon, radius=150)
    
    if not stop_points:
        print(f"          ⚠ No TfL stop points found")
        return {
            'station_name': station_name,
            'station_type': station_type,
            'location': {'latitude': station_lat, 'longitude': station_lon},
            'has_night_service': False,
            'viable_for_night_travel': False,
            'error': 'No stop points found'
        }
    
    # Check each stop point for night service
    has_any_night_service = False
    all_night_tube_lines = set()
    all_night_bus_routes = set()
    all_regular_buses_with_night = set()
    checked_stops = []
    
    for stop in stop_points[:3]:  # Check top 3 closest stops
        stop_id = stop['id']
        print(f"          Checking stop: {stop['name']}")
        
        # Check service types
        night_service_info = check_stop_night_service(stop_id)
        
        if night_service_info['has_night_service']:
            has_any_night_service = True
            
            # Get specific routes
            time.sleep(0.3)  # Be polite to API
            routes = get_night_routes_at_stop(stop_id)
            all_night_tube_lines.update(routes['night_tube_lines'])
            all_night_bus_routes.update(routes['night_bus_routes'])
            all_regular_buses_with_night.update(routes['regular_buses_with_night_service'])
        
        checked_stops.append({
            'stop_id': stop_id,
            'stop_name': stop['name'],
            'modes': stop['modes'],
            'has_night_service': night_service_info['has_night_service'],
            'has_night_service_type': night_service_info['has_night_service_type'],
            'has_night_routes': night_service_info['has_night_routes'],
            'service_types': night_service_info.get('service_types', [])
        })
        
        time.sleep(0.3)  # Be polite to API
    
    # Determine viability
    has_night_tube = len(all_night_tube_lines) > 0
    has_night_bus = len(all_night_bus_routes) > 0
    has_regular_night_bus = len(all_regular_buses_with_night) > 0
    viable = has_any_night_service
    
    # Print summary
    summary_parts = []
    if has_night_tube:
        summary_parts.append(f"Night Tube: {len(all_night_tube_lines)} lines")
    if has_night_bus:
        summary_parts.append(f"Night Bus: {len(all_night_bus_routes)} routes")
    if has_regular_night_bus:
        summary_parts.append(f"24hr Bus: {len(all_regular_buses_with_night)} routes")
    
    if summary_parts:
        print(f"          → {' | '.join(summary_parts)}")
    else:
        print(f"          → NO NIGHT SERVICE")
    
    return {
        'station_name': station_name,
        'station_type': station_type,
        'location': {
            'latitude': station_lat,
            'longitude': station_lon
        },
        'has_night_service': has_any_night_service,
        'night_tube': has_night_tube,
        'night_tube_lines': sorted(list(all_night_tube_lines)),
        'night_bus': has_night_bus,
        'night_bus_routes': sorted(list(all_night_bus_routes)),
        'regular_buses_with_night_service': sorted(list(all_regular_buses_with_night)),
        'checked_stops': checked_stops,
        'viable_for_night_travel': viable
    }

def analyze_route_night_viability(area_slug, club_slug, route_index):
    """
    Analyze if a specific route is viable for night travel.
    
    Args:
        area_slug: Slugified area name
        club_slug: Slugified club name
        route_index: Route index
    
    Returns:
        Dictionary with route viability analysis
    """
    # Load route data
    routes_path = Path(f'data/raw/{area_slug}/routes/{club_slug}_routes.json')
    
    with open(routes_path, 'r') as f:
        routes_data = json.load(f)
    
    route = routes_data['routes'][route_index]
    station_name = route.get('station_name', '')
    station_location = route.get('station_location', {})
    
    # Load the actual station data from filtered_stations
    filtered_stations_path = Path(f'data/raw/{area_slug}/stations/filtered_stations/{club_slug}_stations_filtered.json')
    
    with open(filtered_stations_path, 'r') as f:
        filtered_data = json.load(f)
    
    # Find the matching station in selected_stations
    matching_station = None
    for station in filtered_data.get('selected_stations', []):
        if station.get('location', {}).get('latitude') == station_location.get('latitude'):
            matching_station = station
            break
    
    if not matching_station:
        warnings.warn(f"Could not find station data for route {route_index}")
        return None
    
    # Analyze night service
    night_service = analyze_station_night_service(matching_station)
    
    return {
        'area': area_slug,
        'club': routes_data.get('club', ''),
        'club_location': routes_data.get('club_location', {}),
        'route_index': route_index,
        'station_analysis': night_service,
        'route_viable': night_service['viable_for_night_travel']
    }

def collect_night_service_for_area(area_slug):
    """
    Collect night service data for all routes in an area.
    
    Args:
        area_slug: Slugified area name
    
    Returns:
        Summary statistics
    """
    routes_dir = Path(f'data/raw/{area_slug}/routes')
    
    if not routes_dir.exists():
        return {'total': 0, 'viable': 0, 'not_viable': 0}
    
    # Create night_service directory
    night_service_dir = Path(f'data/raw/{area_slug}/night_service')
    night_service_dir.mkdir(parents=True, exist_ok=True)
    
    route_files = list(routes_dir.glob('*_routes.json'))
    
    total_routes = 0
    viable_routes = 0
    not_viable_routes = 0
    
    for route_file in route_files:
        club_slug = route_file.stem.replace('_routes', '')
        
        with open(route_file, 'r') as f:
            routes_data = json.load(f)
        
        num_routes = len(routes_data.get('routes', []))
        total_routes += num_routes
        
        print(f"    Club: {club_slug}")
        
        for i in range(num_routes):
            print(f"      Route {i}...")
            
            analysis = analyze_route_night_viability(area_slug, club_slug, i)
            
            if analysis:
                # Save analysis
                output_path = night_service_dir / f'{club_slug}_route_{i}_night_service.json'
                with open(output_path, 'w') as f:
                    json.dump(analysis, f, indent=2)
                
                if analysis['route_viable']:
                    viable_routes += 1
                    print(f"        ✓ VIABLE - Night service available")
                else:
                    not_viable_routes += 1
                    print(f"        ✗ NOT VIABLE - No night service")
            
            # Be polite to TfL API
            time.sleep(1)
    
    return {
        'total': total_routes,
        'viable': viable_routes,
        'not_viable': not_viable_routes
    }

# Execute for all areas
AREA_NAMES = ['Dalston', 'Hackney_Wick', 'Bermondsey']

print("="*70)
print("7. Check Night Service for Stations")
print("="*70)
print("Checking which routes have Night Tube or Night Bus service...")
print()

overall_stats = {
    'total': 0,
    'viable': 0,
    'not_viable': 0
}

for area_name in AREA_NAMES:
    area_slug = area_slugify(area_name)
    
    print(f"\n{'─'*70}")
    print(f"📍 Area: {area_name}")
    print(f"{'─'*70}")
    
    stats = collect_night_service_for_area(area_slug)
    
    overall_stats['total'] += stats['total']
    overall_stats['viable'] += stats['viable']
    overall_stats['not_viable'] += stats['not_viable']
    
    print(f"\n  Summary for {area_name}:")
    print(f"    Total routes: {stats['total']}")
    print(f"    Viable (has night service): {stats['viable']}")
    print(f"    Not viable (no night service): {stats['not_viable']}")

# Final summary
print(f"\n{'='*70}")
print(f"Overall Summary")
print(f"{'='*70}")
print(f"Total routes: {overall_stats['total']}")
print(f"Viable routes: {overall_stats['viable']} ({overall_stats['viable']/overall_stats['total']*100:.1f}%)")
print(f"Non-viable routes: {overall_stats['not_viable']} ({overall_stats['not_viable']/overall_stats['total']*100:.1f}%)")
print()
print("⚠️  Routes without night service will be filtered out in NB02 (Data Transformation).")
```